# Bloque B — Control de calidad y partición de las fotos de matriculación

**Proyecto:** Sistema de Reconocimiento Facial para Control de Asistencia (Gimnasio)
**Fase:** CRISP-ML(Q) — Fases 1-2 en versión corta (comprensión y preparación de los datos de matriculación)
**Entregables que cubre:** B.2 (control de calidad automático), B.3 (comprensión de los datos) y B.4 (partición matricular/prueba).

### Por qué existe este notebook
No reentrenamos el modelo, pero las fotos de matriculación siguen siendo los datos de entrada del sistema: una foto borrosa, sin rostro, con otra persona o duplicada en una carpeta empeora el reconocimiento de esa persona **para siempre**, porque queda guardada como referencia. Por eso cada foto pasa por un control antes de matricularse.

### Qué necesita
Una carpeta `matriculacion/` junto a este notebook, con una subcarpeta por persona y las fotos adentro:
```
matriculacion/
    ana_perez/   foto_01.jpg ... foto_28.jpg
    luis_mora/   foto_01.jpg ... foto_28.jpg
```
**Qué produce:** `calidad_fotos.csv` (estado y motivos de cada foto) y `particion_matriculacion.csv` (qué fotos se usan para matricular y cuáles se reservan para probar). Se puede correr con las carpetas que ya tengas y volver a correr cuando lleguen las demás.

## Qué verifica el control de calidad

Cada foto termina en uno de tres estados:
* **ACEPTADA:** pasó todas las revisiones.
* **REVISAR:** hay una duda que decide una persona (por ejemplo, poca luz: el protocolo pide 2 fotos así a propósito).
* **RECHAZADA:** tiene un defecto que la vuelve inútil para matricular.

| Revisión | Resultado si falla |
| --- | --- |
| Archivo ilegible | RECHAZADA |
| No hay rostro, o hay más de uno | RECHAZADA |
| Confianza del detector baja, o rostro muy pequeño | RECHAZADA |
| Rostro casi negro o quemado, o muy borroso | RECHAZADA |
| Duplicado exacto de otra foto de la misma persona | RECHAZADA |
| Poca o mucha luz, poco nítida, cabeza muy inclinada | REVISAR |
| Casi duplicada de otra foto | REVISAR |
| **Parece de otra persona** (su embedding queda lejos del resto de esa carpeta) | REVISAR |

**Limitación honesta:** el sistema no detecta gorras de forma confiable. Por eso, más abajo, hay una *hoja de contacto* para revisar las fotos con tus propios ojos. 28 fotos por persona se revisan en un minuto.

In [ ]:
import os, glob, json, math, hashlib, pickle
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from deepface import DeepFace
from deepface.modules.verification import find_cosine_distance, find_threshold

# --- Modelo y detector validados en el Bloque A ---
MODELO = "Facenet512"
DETECTOR = "mtcnn"
UMBRAL = find_threshold(MODELO, "cosine")   # 0.30 para Facenet512 + coseno

# --- Rutas ---
CARPETA_MATRICULACION = "matriculacion"
CSV_CALIDAD = "calidad_fotos.csv"
CSV_PARTICION = "particion_matriculacion.csv"
CACHE = "cache_calidad.pkl"   # guarda los embeddings ya calculados para no repetir el trabajo

# --- Umbrales (valores iniciales: se ajustan mirando los histogramas más abajo) ---
MAX_LADO = 1280          # las fotos grandes del celular se reducen a este lado máximo antes de detectar
MIN_CONFIANZA = 0.90     # confianza mínima del detector mtcnn
MIN_LADO_ROSTRO = 100    # píxeles mínimos del rostro (lado menor) tras reducir la foto
BRILLO_RECHAZO = (35, 235)   # brillo medio del rostro (0-255) fuera de este rango: RECHAZADA
BRILLO_REVISAR = (70, 200)   # fuera de este rango: REVISAR (poca o mucha luz)
NITIDEZ_RECHAZO = 15     # varianza del Laplaciano del rostro: menos de esto es una foto muy borrosa
NITIDEZ_REVISAR = 40     # menos de esto: REVISAR
ANGULO_REVISAR = 30      # grados de inclinación de la línea de los ojos
HAMMING_CASI_DUP = 4     # bits de diferencia (de 64) para considerar una foto casi duplicada
DIST_OTRA_PERSONA = 0.40 # distancia coseno al resto de la carpeta para sospechar de otra persona
MIN_FOTOS_CONSISTENCIA = 5   # mínimo de fotos válidas para poder comparar contra el resto
MIN_ACEPTADAS_POR_PERSONA = 20

print(f"Modelo: {MODELO} | Detector: {DETECTOR} | Umbral de reconocimiento: {UMBRAL}")

## Funciones de ayuda

**Concepto — nitidez:** se mide con la *varianza del Laplaciano*. El Laplaciano resalta los bordes de una imagen; una foto nítida tiene bordes marcados (varianza alta) y una borrosa los tiene suavizados (varianza baja). Se calcula solo sobre el recorte del rostro, a un tamaño fijo, para que sea comparable entre fotos.

**Concepto — hash perceptual:** se reduce la foto a 8×8 píxeles en gris y se marca cuáles están sobre el promedio. Dos fotos casi idénticas dan huellas casi iguales, aunque el archivo no sea el mismo.

In [ ]:
def cargar_imagen(ruta):
    """Lee la foto (OpenCV respeta la rotación EXIF del celular) y la reduce si es muy grande."""
    img = cv2.imread(ruta)
    if img is None:
        return None
    h, w = img.shape[:2]
    escala = min(1.0, MAX_LADO / max(h, w))
    if escala < 1.0:
        img = cv2.resize(img, (int(w * escala), int(h * escala)), interpolation=cv2.INTER_AREA)
    return img

def recortar(img, area):
    x, y, w, h = max(area["x"], 0), max(area["y"], 0), area["w"], area["h"]
    return img[y:y + h, x:x + w]

def medir_nitidez(recorte_bgr):
    gris = cv2.cvtColor(cv2.resize(recorte_bgr, (160, 160)), cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gris, cv2.CV_64F).var())

def medir_angulo(area):
    """Inclinación (en grados) de la línea que une los ojos. 0 = cabeza derecha."""
    ojo_i, ojo_d = area.get("left_eye"), area.get("right_eye")
    if not ojo_i or not ojo_d:
        return 0.0
    ang = math.degrees(math.atan2(ojo_d[1] - ojo_i[1], ojo_d[0] - ojo_i[0]))
    if ang > 90:
        ang -= 180
    if ang < -90:
        ang += 180
    return float(ang)

def hash_perceptual(img):
    chico = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (8, 8), interpolation=cv2.INTER_AREA)
    return (chico > chico.mean()).flatten()

def md5_archivo(ruta):
    with open(ruta, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def normalizar(v):
    v = np.asarray(v, dtype=float)
    return v / np.linalg.norm(v)

## Paso 1: Analizar cada foto

Una sola llamada a `DeepFace.represent()` entrega, para cada rostro encontrado, su **embedding**, la **zona del rostro** y la **confianza** del detector. Con eso hacemos casi todas las revisiones sin repetir trabajo.

Si algo falla por un motivo que **no** es "no hay rostro", el notebook se detiene y muestra el error real, en vez de disfrazarlo como un problema de la foto (lo aprendimos en el Bloque A).

In [ ]:
def listar_fotos():
    filas = []
    for carpeta in sorted(glob.glob(os.path.join(CARPETA_MATRICULACION, "*"))):
        if not os.path.isdir(carpeta):
            continue
        persona = os.path.basename(carpeta)
        for ruta in sorted(glob.glob(os.path.join(carpeta, "*"))):
            if ruta.lower().endswith((".jpg", ".jpeg", ".png")):
                filas.append((persona, os.path.basename(ruta), ruta))
    return filas

cache = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) else {}

def analizar_foto(persona, foto, ruta):
    f = dict(persona=persona, foto=foto, ruta=ruta, n_rostros=0, confianza=np.nan, lado_rostro=np.nan,
             brillo=np.nan, nitidez=np.nan, angulo=np.nan, md5=None, rechazo=[], revisar=[])
    f["md5"] = md5_archivo(ruta)
    img = cargar_imagen(ruta)
    if img is None:
        f["rechazo"].append("archivo ilegible")
        return f, None, None
    f_hash = hash_perceptual(img)
    clave = (ruta, os.path.getmtime(ruta), os.path.getsize(ruta))
    if clave not in cache:
        try:
            cache[clave] = DeepFace.represent(img_path=img, model_name=MODELO, detector_backend=DETECTOR,
                                              enforce_detection=True)
        except ValueError as e:
            real = str(e.__cause__) if e.__cause__ is not None else str(e)
            if "could not be detected" in real.lower():
                cache[clave] = []
            else:
                raise   # error real (no es culpa de la foto): no lo escondemos
    rostros = cache[clave]
    f["n_rostros"] = len(rostros)
    if len(rostros) == 0:
        f["rechazo"].append("no se detecta ningún rostro")
        return f, None, f_hash
    if len(rostros) > 1:
        f["rechazo"].append(f"hay {len(rostros)} rostros en la foto")
        return f, None, f_hash
    r = rostros[0]
    area = r["facial_area"]
    recorte = recortar(img, area)
    f["confianza"] = float(r.get("face_confidence") or 0)
    f["lado_rostro"] = int(min(area["w"], area["h"]))
    f["brillo"] = float(cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY).mean())
    f["nitidez"] = medir_nitidez(recorte)
    f["angulo"] = medir_angulo(area)

    if f["confianza"] < MIN_CONFIANZA:
        f["rechazo"].append(f"confianza del detector baja ({f['confianza']:.2f})")
    if f["lado_rostro"] < MIN_LADO_ROSTRO:
        f["rechazo"].append(f"rostro muy pequeño ({f['lado_rostro']} px)")
    if f["brillo"] < BRILLO_RECHAZO[0] or f["brillo"] > BRILLO_RECHAZO[1]:
        f["rechazo"].append(f"rostro casi negro o quemado (brillo {f['brillo']:.0f})")
    elif f["brillo"] < BRILLO_REVISAR[0]:
        f["revisar"].append(f"poca luz (brillo {f['brillo']:.0f})")
    elif f["brillo"] > BRILLO_REVISAR[1]:
        f["revisar"].append(f"mucha luz (brillo {f['brillo']:.0f})")
    if f["nitidez"] < NITIDEZ_RECHAZO:
        f["rechazo"].append(f"muy borrosa (nitidez {f['nitidez']:.0f})")
    elif f["nitidez"] < NITIDEZ_REVISAR:
        f["revisar"].append(f"poco nítida (nitidez {f['nitidez']:.0f})")
    if abs(f["angulo"]) > ANGULO_REVISAR:
        f["revisar"].append(f"cabeza muy inclinada ({f['angulo']:.0f}°)")
    return f, np.array(r["embedding"]), f_hash

listado = listar_fotos()
assert listado, f"No encontré fotos en '{CARPETA_MATRICULACION}/<persona>/'. Revisa la estructura de carpetas."
print(f"Analizando {len(listado)} fotos de {len({p for p, _, _ in listado})} persona(s)...")

filas, embeddings, hashes = [], {}, {}
for i, (persona, foto, ruta) in enumerate(listado, 1):
    fila, emb, hsh = analizar_foto(persona, foto, ruta)
    filas.append(fila)
    if emb is not None:
        embeddings[ruta] = emb
    if hsh is not None:
        hashes[ruta] = hsh
    if i % 20 == 0:
        print(f"  {i}/{len(listado)}")

pickle.dump(cache, open(CACHE, "wb"))
df = pd.DataFrame(filas)
print("Análisis individual terminado.")

## Paso 2: Duplicados y fotos que parecen de otra persona

Estas dos revisiones comparan las fotos **entre sí, dentro de cada carpeta**.

**Consistencia de identidad:** para cada foto se calcula la distancia coseno entre su embedding y el **promedio de las demás fotos de esa persona** (sin incluirla a ella misma; así una foto equivocada no contamina el promedio). Si queda muy lejos, probablemente alguien mezcló una foto de otra persona en esa carpeta. Se necesita un mínimo de fotos válidas para que el promedio signifique algo.

In [ ]:
# --- Duplicados dentro de cada persona ---
for persona, g in df.groupby("persona"):
    vistos_md5, vistas = {}, []
    for idx in g.index:
        ruta, foto, md5 = df.at[idx, "ruta"], df.at[idx, "foto"], df.at[idx, "md5"]
        if md5 in vistos_md5:
            df.at[idx, "rechazo"].append(f"duplicado exacto de {vistos_md5[md5]}")
            continue
        vistos_md5[md5] = foto
        if ruta in hashes:
            for ruta_ant, foto_ant in vistas:
                if int((hashes[ruta] != hashes[ruta_ant]).sum()) <= HAMMING_CASI_DUP:
                    df.at[idx, "revisar"].append(f"casi duplicada de {foto_ant}")
                    break
            vistas.append((ruta, foto))

# --- Consistencia de identidad (distancia al resto de la carpeta) ---
df["distancia_identidad"] = np.nan
for persona, g in df.groupby("persona"):
    validas = [i for i in g.index if df.at[i, "ruta"] in embeddings and not df.at[i, "rechazo"]]
    if len(validas) < MIN_FOTOS_CONSISTENCIA:
        print(f"[AVISO] {persona}: solo {len(validas)} fotos válidas; no se puede comprobar la identidad (mínimo {MIN_FOTOS_CONSISTENCIA}).")
        continue
    normas = {i: normalizar(embeddings[df.at[i, "ruta"]]) for i in validas}
    for i in validas:
        resto = np.mean([normas[j] for j in validas if j != i], axis=0)
        d = float(find_cosine_distance(normas[i], resto))
        df.at[i, "distancia_identidad"] = d
        if d > DIST_OTRA_PERSONA:
            df.at[i, "revisar"].append(f"posible foto de otra persona (distancia {d:.2f})")
print("Revisiones entre fotos terminadas.")

## Paso 3: Estado final de cada foto

Aquí también puedes dejar **decisiones manuales** después de mirar las hojas de contacto del Paso 5. Escribe la clave como `"persona/foto"` y el estado que quieres. Por ejemplo:
```python
DECISIONES_MANUALES = {"ana_perez/foto_07.jpg": "ACEPTADA", "luis_mora/foto_12.jpg": "RECHAZADA"}
```

In [ ]:
DECISIONES_MANUALES = {}   # <- se llena después de revisar las hojas de contacto

def estado_foto(r):
    if r["rechazo"]:
        return "RECHAZADA"
    if r["revisar"]:
        return "REVISAR"
    return "ACEPTADA"

df["estado"] = df.apply(estado_foto, axis=1)
df["motivos"] = df.apply(lambda r: "; ".join(r["rechazo"] + r["revisar"]), axis=1)

for clave, estado in DECISIONES_MANUALES.items():
    persona, foto = clave.split("/")
    mascara = (df["persona"] == persona) & (df["foto"] == foto)
    assert mascara.any(), f"La decisión manual '{clave}' no corresponde a ninguna foto."
    assert estado in ("ACEPTADA", "REVISAR", "RECHAZADA"), f"Estado inválido en '{clave}'"
    df.loc[mascara, "estado"] = estado
    df.loc[mascara, "motivos"] += " [decisión manual]"

columnas = ["persona", "foto", "estado", "motivos", "n_rostros", "confianza", "lado_rostro", "brillo",
            "nitidez", "angulo", "distancia_identidad"]
df[columnas].to_csv(CSV_CALIDAD, index=False, encoding="utf-8")
print(f"Guardado: {CSV_CALIDAD}")
df[df["estado"] != "ACEPTADA"][["persona", "foto", "estado", "motivos"]]

## Paso 4: Comprensión de los datos (B.3)

Resumen por persona y distribución de las medidas. Los histogramas sirven para **ajustar los umbrales**: si casi todas tus fotos buenas quedan por debajo de una línea roja, esa línea está mal puesta.

In [ ]:
resumen = (df.pivot_table(index="persona", columns="estado", values="foto", aggfunc="count", fill_value=0)
             .reindex(columns=["ACEPTADA", "REVISAR", "RECHAZADA"], fill_value=0))
resumen["total"] = resumen.sum(axis=1)
resumen["suficiente"] = np.where(resumen["ACEPTADA"] >= MIN_ACEPTADAS_POR_PERSONA, "sí", "NO")
display(resumen)

ax = resumen[["ACEPTADA", "REVISAR", "RECHAZADA"]].plot(kind="bar", stacked=True, figsize=(10, 4),
                                                         color=["#4caf50", "#ff9800", "#e53935"])
ax.axhline(MIN_ACEPTADAS_POR_PERSONA, color="black", linestyle="--", linewidth=1)
ax.set_title(f"Estado de las fotos por persona (la línea marca el mínimo de {MIN_ACEPTADAS_POR_PERSONA} aceptadas)")
ax.set_ylabel("Cantidad de fotos")
plt.tight_layout()
plt.show()

faltan = resumen[resumen["suficiente"] == "NO"]
if len(faltan):
    print("Personas con menos fotos aceptadas de las necesarias:", list(faltan.index))

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 3.6))
medidas = [("brillo", "Brillo medio del rostro", [BRILLO_REVISAR[0], BRILLO_REVISAR[1]]),
           ("nitidez", "Nitidez del rostro (varianza del Laplaciano)", [NITIDEZ_RECHAZO, NITIDEZ_REVISAR]),
           ("distancia_identidad", "Distancia al resto de la carpeta", [DIST_OTRA_PERSONA])]
for ax, (col, titulo, lineas) in zip(ejes, medidas):
    datos = df[col].dropna()
    if len(datos):
        ax.hist(datos, bins=15, color="#5c6bc0")
    for x in lineas:
        ax.axvline(x, color="red", linestyle="--")
    ax.set_title(titulo, fontsize=10)
plt.tight_layout()
plt.show()

## Paso 5: Hoja de contacto (revisión visual)

Cada persona se muestra en una cuadrícula. El título de cada foto va **verde** (aceptada), **naranja** (revisar) o **rojo** (rechazada). Mira específicamente: **gorras**, rostros tapados, y fotos que claramente no son de esa persona. Si algo no coincide con el estado automático, corrígelo en `DECISIONES_MANUALES` (Paso 3) y vuelve a correr desde esa celda.

In [ ]:
COLORES = {"ACEPTADA": "#2e7d32", "REVISAR": "#ef6c00", "RECHAZADA": "#c62828"}

def hoja_de_contacto(persona, columnas=7):
    g = df[df["persona"] == persona]
    filas = math.ceil(len(g) / columnas)
    fig, ejes = plt.subplots(filas, columnas, figsize=(2.2 * columnas, 2.5 * filas))
    ejes = np.atleast_1d(ejes).flatten()
    for ax in ejes:
        ax.axis("off")
    for ax, (_, r) in zip(ejes, g.iterrows()):
        img = cv2.imread(r["ruta"])
        if img is not None:
            h, w = img.shape[:2]
            e = 240 / max(h, w)
            ax.imshow(cv2.cvtColor(cv2.resize(img, (int(w * e), int(h * e))), cv2.COLOR_BGR2RGB))
        ax.set_title(r["foto"], fontsize=8, color=COLORES[r["estado"]])
    fig.suptitle(persona, fontsize=13)
    plt.tight_layout()
    plt.show()

for persona in sorted(df["persona"].unique()):
    hoja_de_contacto(persona)

## Paso 6: Partición matricular / probar (B.4)

**Concepto:** si midiéramos la exactitud con las mismas fotos que usamos para matricular, el resultado sería trampa (el sistema "ya vio" esas fotos). Por eso, de las fotos aceptadas de cada persona, reservamos una parte que **nunca se matricula**: sirve solo para probar. La propuesta es **20 para matricular y 8 para probar** de las 28 (proporción 8/28), con una semilla fija para que el reparto sea siempre el mismo.

Solo entran las fotos **ACEPTADAS**. Si quieres incluir las que están en REVISAR, primero decide cada una en `DECISIONES_MANUALES`.

In [ ]:
SEMILLA = 42
FRACCION_PRUEBA = 8 / 28

filas_part = []
for persona, g in df[df["estado"] == "ACEPTADA"].groupby("persona"):
    g = g.sample(frac=1.0, random_state=SEMILLA)
    n_prueba = max(1, round(len(g) * FRACCION_PRUEBA)) if len(g) >= 2 else 0
    for k, (_, r) in enumerate(g.iterrows()):
        filas_part.append(dict(persona=persona, foto=r["foto"], ruta=r["ruta"],
                               conjunto="prueba" if k < n_prueba else "matricular"))
particion = pd.DataFrame(filas_part).sort_values(["persona", "conjunto", "foto"])
particion.to_csv(CSV_PARTICION, index=False, encoding="utf-8")
print(f"Guardado: {CSV_PARTICION}")
display(particion.pivot_table(index="persona", columns="conjunto", values="foto", aggfunc="count", fill_value=0))

## Siguiente paso

Con `particion_matriculacion.csv` ya está definido **qué fotos se matriculan y cuáles se reservan**. El **Bloque C** usa justamente esa partición: matricula a las 13 personas con sus fotos de `matricular` (con las funciones del Bloque A) y mide el reconocimiento con las de `prueba`.

Puedes volver a correr este notebook cada vez que lleguen carpetas nuevas: los embeddings ya calculados se guardan en `cache_calidad.pkl` y no se repiten.